# Generate output and evaluate models on held-out test set

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import (
    BEST_PARAM_DICT,
    col_config_sensitivity,
    export_data,
    get_test_df_path,
    get_df_path,
)
from src.residual_func import get_X_y
from src.Regressor import ResidualRegressor, IndependentRegressor
from sklearn.metrics import r2_score
import pandas as pd

In [ ]:
train_data_dir = BASE_PATH / "data"
test_data_dir = BASE_PATH / "data/test_data_toy"
model_dir = BASE_PATH / "models"
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}

In [ ]:
def get_full_reduced_scores(residual_setup, pred_df, y_true):
    if residual_setup not in ["SQT", "ST", "QT", "T"]:
        raise ValueError(f"Unknown residual_setup: {residual_setup}")
    ## Get full score
    full_name = "pred_stage3_total"
    full_score = r2_score(y_true=y_true, y_pred=pred_df[full_name])
    if residual_setup == "T":
        return {residual_setup: full_score}
    ## Get reduced score
    # Stage 2 for full-reduced SQT/SQ and QT/Q models-- stage 1 for ST/S models
    reduced_name = (
        "pred_stage2_total" if residual_setup in ["SQT", "QT"] else "pred_stage1"
    )
    reduced_setup = residual_setup[:-1]
    reduced_score = r2_score(y_true=y_true, y_pred=pred_df[reduced_name])
    return {residual_setup: full_score, reduced_setup: reduced_score}

In [ ]:
result_list = []
for residual_setup, sub_dict in BEST_PARAM_DICT.items():
    for impact_type, param_dict in sub_dict.items():
        quest_cols = col_config_sensitivity[f"{impact_type}_quest"]
        ## Create and train model
        param_dict = {
            **param_dict,
            "impact_type": impact_type,
            "quest_cols": quest_cols,
            **SHARED_DICT,
        }
        model_config = {
            k: v for k, v in param_dict.items() if k not in ["llm_model", "temp"]
        }
        ## PREP TRAIN/TEST
        # train
        train_df_path = get_df_path(
            df_dir=train_data_dir,
            llm_model=param_dict["llm_model"],
            agg_type=param_dict["agg_type"],
            desc_seg_type=param_dict["desc_seg_type"],
            impact_type=impact_type,
            temp=param_dict["temp"],
        )
        train_df = pd.read_parquet(train_df_path)
        X_train, y_train = get_X_y(df=train_df, config_dict=param_dict, scale=False)
        # test
        test_df_path = get_test_df_path(
            df_dir=test_data_dir,
            llm_model=param_dict["llm_model"],
            agg_type=param_dict["agg_type"],
            desc_seg_type=param_dict["desc_seg_type"],
            impact_type=impact_type,
            temp=param_dict["temp"],
        )
        test_df = pd.read_parquet(test_df_path)
        X_test, y_test = get_X_y(df=test_df, config_dict=param_dict, scale=False)
        ## FIT/EXPORT MODEL
        if param_dict["num_layers"] == 1:
            model = IndependentRegressor(**model_config)
        else:
            model = ResidualRegressor(**model_config)
        model.fit(X_train, y_train, shuffle=None, perm_dir=None)
        export_data(
            data_to_export=model,
            export_path=model_dir / f"{residual_setup}_{impact_type}.joblib",
            remove_existing=True,
        )
        ## GET PREDICTIONS/SCORES
        # Train
        y_pred_train_df = model.predict_all(X_train)
        train_score_dict = get_full_reduced_scores(
            residual_setup=residual_setup, pred_df=y_pred_train_df, y_true=y_train
        )
        # Test
        y_pred_test_df = model.predict_all(X_test)
        test_score_dict = get_full_reduced_scores(
            residual_setup=residual_setup, pred_df=y_pred_test_df, y_true=y_test
        )
        for set_up in train_score_dict.keys():
            assert (
                set_up in test_score_dict
            ), f"Set-up type {set_up} not found in test scores"
            # Append results
            result_list.append(
                {
                    "residual_setup": set_up,
                    "impact_type": impact_type,
                    "r2_train": round(train_score_dict[set_up], 4),
                    "r2_test": round(test_score_dict[set_up], 4),
                }
            )

result_df = pd.DataFrame(result_list)

In [ ]:
setup_order = ["SQT", "SQ", "ST", "QT", "S", "Q", "T"]
impact_order = ["parent", "self"]

result_df["residual_setup"] = pd.Categorical(
    result_df["residual_setup"], setup_order, ordered=True
)
result_df["impact_type"] = pd.Categorical(
    result_df["impact_type"], impact_order, ordered=True
)

result_df = result_df.sort_values(["residual_setup", "impact_type"]).reset_index(
    drop=True
)
result_df